# 🫀 rPPG-Toolbox: DeepPhys Training & Evaluation on MMPD Dataset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drozanes/rPPG-Toolbox/blob/main/MMPD_DeepPhys_Colab.ipynb)

This notebook runs **DeepPhys** training and evaluation end-to-end on the complete **MMPD** benchmark dataset (all 33 subjects, 660 raw video files) using Google Colab.

### ⚙️ Recommended Colab Runtime Setup:
- **Hardware Accelerator:** `T4 GPU` (best balance of compute units: only ~1.5 - 2 CU/hr vs ~12 CU/hr on A100)
- **High-RAM:** **Toggle ON** (gives 53.5 GB RAM and 8 vCPUs, speeding up CPU data loading 4x)

### Key Engineering Highlights:
- **Zero Preprocessing Disk Footprint:** Uses Online Memory Streaming (`MMPDLoader` with LRU caching) to completely bypass the ~940 GB disk requirement.
- **Real-Time Checkpoint Sync to Google Drive:** Every epoch's checkpoint (`.pth`) is immediately mirrored to your Google Drive (`MyDrive/rPPG_Checkpoints`), ensuring you never lose progress if Colab disconnects.
- **Direct Download to Local Disk:** Download any checkpoint or the complete model archive directly to your computer.
- **Resume Support:** Seamlessly resume training from an existing checkpoint (e.g. `MMPD_DeepPhys_Online_Epoch5.pth`).
- **Validation Tracking & Early Stopping:** Tracks validation loss each epoch (`patience=3`) and saves the best model.
- **Automated Benchmark Recording:** Best checkpoint is automatically evaluated on the test set (MAE, RMSE, MAPE, Pearson $r$, SNR) and recorded into `rPPG_results_database.csv`.

In [ ]:
# 1. Verify GPU Allocation & System Resources
!nvidia-smi
import psutil
print(f"CPU Cores: {psutil.cpu_count(logical=True)}, System RAM: {psutil.virtual_memory().total / (1024**3):.1f} GB")

In [ ]:
# 2. (Highly Recommended) Mount Google Drive for Real-Time Model Saving
# Mounting Google Drive ensures all trained models (.pth) and benchmark results are saved directly
# to your Google Drive ('MyDrive/rPPG_Checkpoints') in real time at the end of each epoch!
from google.colab import drive
import os

drive.mount('/content/drive')
GDRIVE_DIR = "/content/drive/MyDrive/rPPG_Checkpoints"
os.makedirs(GDRIVE_DIR, exist_ok=True)
print(f"✅ Google Drive mounted! Real-time backups will be saved to: {GDRIVE_DIR}")

In [ ]:
# 3. Clone the Repository and Install Dependencies
!git clone https://github.com/drozanes/rPPG-Toolbox.git
%cd rPPG-Toolbox

# Install required dependencies
!pip install -q yacs mat73 huggingface_hub wandb pyyaml scipy opencv-python tqdm pandas matplotlib scikit-learn


In [ ]:
# 4. Disk Space Verification & Proactive Cleanup
import shutil
total, used, free = shutil.disk_usage("/")
print(f"Colab Root Disk: Total = {total/(1024**3):.1f} GB, Usable Free = {free/(1024**3):.1f} GB")

# If free space is below 55 GB, clear package caches for extra headroom
if free / (1024**3) < 55:
    print("Cleaning package cache for extra disk headroom...")
    !apt-get clean > /dev/null 2>&1
    !rm -rf /root/.cache/pip
    _, _, free = shutil.disk_usage("/")
    print(f"Updated Free Disk: {free/(1024**3):.1f} GB")

In [ ]:
# 5. Download Raw MMPD Mini Dataset (~47.3 GB, 660 files) from Hugging Face
# Uses token authentication for the gated WeiQian98/mini_MMPD repository
import os, glob, getpass
from huggingface_hub import snapshot_download

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    print("Please enter your Hugging Face Access Token (read access to WeiQian98/mini_MMPD):")
    HF_TOKEN = getpass.getpass("HF Token: ").strip()

REPO_ID = "WeiQian98/mini_MMPD"
data_dir = "./MMPD_dataset"
os.makedirs(data_dir, exist_ok=True)

print(f"Downloading all 660 .mat files from {REPO_ID} (takes ~10-12 minutes)...")
snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=data_dir,
    token=HF_TOKEN,
    ignore_patterns=[".DS_Store"],
    resume_download=True
)

mat_files = glob.glob(f"{data_dir}/**/*.mat", recursive=True)
print(f"✅ Download verification: {len(mat_files)} / 660 .mat files successfully downloaded!")
assert len(mat_files) == 660, f"Expected 660 files, found {len(mat_files)}"


In [ ]:
# 6. (Recommended) Log in to Weights & Biases for live metrics tracking
# Click the link displayed (https://wandb.ai/authorize), copy your API key and paste it here.
# Training loss, validation loss, and final test metrics will stream live to your W&B dashboard.
import wandb
wandb.login()


In [ ]:
# 7. (Optional) Provide Existing Checkpoint to Resume Training or Run Test Evaluation
# Checks Google Drive first, or prompts for upload from your local computer.
import os, shutil
checkpoint_file = "MMPD_DeepPhys_Online_Epoch5.pth"
gdrive_checkpoint = f"/content/drive/MyDrive/rPPG_Checkpoints/{checkpoint_file}"

if os.path.exists(gdrive_checkpoint) and not os.path.exists(checkpoint_file):
    print(f"Found checkpoint in Google Drive! Copying from: {gdrive_checkpoint}")
    shutil.copy2(gdrive_checkpoint, checkpoint_file)
elif not os.path.exists(checkpoint_file):
    from google.colab import files
    print(f"Please upload '{checkpoint_file}' from your local disk:")
    uploaded = files.upload()

if os.path.exists(checkpoint_file):
    print(f"✅ Checkpoint {checkpoint_file} ready to use ({os.path.getsize(checkpoint_file)/(1024**2):.1f} MB)!")


In [ ]:
# 8A. RESUME DeepPhys Online Training from Epoch 5 (Continues from Epoch 6)
# - Real-time checkpoint mirror: Saves each new epoch to /content/drive/MyDrive/rPPG_Checkpoints/
# - Automatically fast-forwards scheduler and verifies baseline validation loss (0.9759).
# - Multi-worker CPU streaming with --num_workers 4 for 4x faster throughput.
!python main.py --config_file configs/train_configs/MMPD_DEEPPHYS_ONLINE.yaml --device cuda --num_workers 4 --resume MMPD_DeepPhys_Online_Epoch5.pth --backup_dir /content/drive/MyDrive/rPPG_Checkpoints


In [ ]:
# 8B. (Alternative) Evaluate Epoch 5 Model Directly on Test Set (Subjects 30-33, 800 chunks)
# Takes ~2 minutes. Computes MAE, RMSE, Pearson r, SNR and saves to database.
# !python main.py --config_file configs/train_configs/MMPD_DEEPPHYS_ONLINE.yaml --device cuda --num_workers 4 --toolbox_mode only_test --model_path MMPD_DeepPhys_Online_Epoch5.pth --backup_dir /content/drive/MyDrive/rPPG_Checkpoints


In [ ]:
# 9. Display Updated Benchmark Results
import os, pandas as pd
if os.path.exists("rPPG_results_database.csv"):
    df = pd.read_csv("rPPG_results_database.csv")
    print("\n=== Benchmark Database (MMPD Entries) ===")
    display(df[df["Dataset"] == "MMPD"])
else:
    print("Results database not found.")

In [ ]:
# 10. Download Checkpoints Directly to Local Computer (Mac Downloads folder)
from google.colab import files
import glob, os

# List available saved checkpoints
pth_files = sorted(glob.glob("runs/**/*.pth", recursive=True) + glob.glob("PreTrainedModels/**/*.pth", recursive=True))
print(f"Found {len(pth_files)} checkpoints:")
for p in pth_files:
    print(f" - {p} ({os.path.getsize(p)/(1024**2):.1f} MB)")

# Download the best/latest checkpoint to your local computer
if pth_files:
    latest = pth_files[-1]
    print(f"\nDownloading {latest} to your local disk...")
    files.download(latest)


In [ ]:
# 11. Bundle All Trained Weights, Runs & Results and Download Zip
!zip -q -r mmpd_deepphys_trained_model.zip PreTrainedModels/ runs/ rPPG_results_database.csv
from google.colab import files
files.download("mmpd_deepphys_trained_model.zip")
print("✅ Full archive download initiated for mmpd_deepphys_trained_model.zip!")